In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid


import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# Convert MNIST Image Files into a Tensor of 4-Dimensions (# of images, Height, Width, Color Channel)
transform = transforms.ToTensor()

In [4]:
# Train Data
train_data = datasets.MNIST(root='./cnn_data', train=True, download=True, transform=transform)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|███████████████████████████| 9912422/9912422 [00:00<00:00, 45621168.35it/s]


Extracting ./cnn_data/MNIST/raw/train-images-idx3-ubyte.gz to ./cnn_data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|████████████████████████████████| 28881/28881 [00:00<00:00, 1547091.20it/s]


Extracting ./cnn_data/MNIST/raw/train-labels-idx1-ubyte.gz to ./cnn_data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|████████████████████████████| 1648877/1648877 [00:00<00:00, 5844673.75it/s]


Extracting ./cnn_data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./cnn_data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████████████████████████████| 4542/4542 [00:00<00:00, 1097127.89it/s]

Extracting ./cnn_data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./cnn_data/MNIST/raw



In [5]:
#Test Data
test_data = datasets.MNIST(root='./cnn_data', train=False, download=True, transform=transform)

In [6]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./cnn_data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [7]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [8]:
pwd

'/Users/ibrahimcosar/Desktop/Data Analysis/PytorchStarter'

In [9]:
# Create a small batch size for images ... let's say 10
train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

In [10]:
# Define Our CNN Model
# Describe convolutional layer and what it's doing (2 convolutional layers)

conv1 = nn.Conv2d(1, 6, 3, 1)
conv2 = nn.Conv2d(6, 16, 3, 1)

In [11]:
# Grab 1 MNIST record/image
for i, (X_Train, y_train) in enumerate(train_data):
    break

In [12]:
X_Train.shape

torch.Size([1, 28, 28])

In [13]:
x = X_Train.view(1,1,28,28)

In [14]:
# Perform our first convolution
x = F.relu(conv1(x)) # Rectified Linear Unit for our activation function

[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


In [15]:
# 1 single image, 6 is the filters we asked for, 26*26 is the image
x.shape

torch.Size([1, 6, 26, 26])

In [16]:
# pass thru the pooling layer
x = F.max_pool2d(x, 2, 2) #kernal of 2 and stride of 2

In [17]:
x.shape

torch.Size([1, 6, 13, 13])

In [19]:
# Do our second convolutional layer
x = F.relu(conv2(x))

In [20]:
x.shape

torch.Size([1, 16, 11, 11])

In [21]:
# Another pooling layer
x = F.max_pool2d(x,2,2)

In [22]:
x.shape

torch.Size([1, 16, 5, 5])

In [23]:
#we have lost data in the pooling, so we have to round down

In [26]:
#Model Class
class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 3, 1)
        self.conv2 = nn.Conv2d(6, 16, 3, 1)
        # Fully Connected Layer
        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(x,2,2)
        # Second Pass
        X = F.relu(self.conv2(X))
        X = F.max_pool2d(x,2,2)
        
        
        # Re-View to flatten it out
        X = X.view(-1, 16*5*5) # negative one so that we can vary the batch size
        
        # Fully Connected Layers
        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)

In [27]:
# Create an Instance of our Model
torch.manual_seed(41)
model = ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [28]:
# Loss Function Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) #Smaller the Learning Rate, longer it's going to take to train.

In [29]:
import time
start_time = time.time()

# Create Variables To Track Things
epochs = 5
train_losses = []
test_losses = []
train_correct = []
test_correct = []


# For Loop of Epochs
for i in range(epochs):
    trn_corr = 0
    tst_corr = 0
    

    # Train
    for b,(X_train, y_train) in enumerate(train_loader):
        b+=1 #start our batches at 1
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train) #how off are we?
        
        predicted = torch.max(y_pred.data, 1)[1] # add up the number of correct predicitons. Indexed off the first point.
        batch_corr = (predicted == y_train).sum() # how many we got correct from this batch?
        trn_corr += batch_corr #keep track as we go along in training.
        
        # Update our parameters
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        
        # Print out some results
        if b%600 == 0:
            print(f'Epoch: {i} Batch: {b} Loss: {loss.item()}')
        
    train_losses.append(loss)
    train_correct.append(trn_corr)
    
    # Test

    with torch.no_grad(): #No gradient so we don't update our weights and biases with tests
        for b,(X_test, y_test) in enumerate(test_loader):
            y_val = model(X_test)
            predicted = torch.max(y_val.data, 1)[1] #Adding up correct predictions
            tst_corr += (predicted == y_test).sum()

    loss = criterion(y_val, y_test)
    test_losses.append(loss)
    test_correct.apped(tst_corr)

            
            
current_time = time.time()
total = current_time - start_time
print(f'Training Took: {total/60} minutes!')

RuntimeError: Given groups=1, weight of size [16, 6, 3, 3], expected input[1, 16, 2, 2] to have 6 channels, but got 16 channels instead